In [1]:
import os
import pandas as pd
import pyodbc

SERVER = "10.147.17.185"
PORT = 1433
USERNAME = "cmpcuser"
PASSWORD = os.getenv("OTMS_SQL_PASSWORD_DEV")
DRIVER = "{ODBC Driver 17 for SQL Server}"

DB = "cmpc_20240925_093000"
SCHEMA = "dbo"
TABLE = "ypf_alarms"

conn_str = (
    f"DRIVER={DRIVER};"
    f"SERVER={SERVER},{PORT};"
    f"DATABASE={DB};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "TrustServerCertificate=yes;"
)
conn = pyodbc.connect(conn_str)
print("Conectado.")

Conectado.


In [2]:
DAYS_WINDOW = 30

q = f"""
WITH mx AS (
  SELECT MAX(ALARMDATETIME) AS mx_t
  FROM [{DB}].[{SCHEMA}].[{TABLE}]
),
minute_counts AS (
  SELECT
    DATEADD(minute, DATEDIFF(minute, 0, a.ALARMDATETIME), 0) AS [minute],
    COUNT(*) AS alarms
  FROM [{DB}].[{SCHEMA}].[{TABLE}] a
  CROSS JOIN mx
  WHERE a.ALARMDATETIME >= DATEADD(day, -{DAYS_WINDOW}, mx.mx_t)
  GROUP BY DATEADD(minute, DATEDIFF(minute, 0, a.ALARMDATETIME), 0)
)
SELECT alarms
FROM minute_counts;
"""

df_min = pd.read_sql(q, conn)
x = df_min["alarms"].to_numpy()

rate_p95  = float(pd.Series(x).quantile(0.95))
rate_p99  = float(pd.Series(x).quantile(0.99))
rate_p999 = float(pd.Series(x).quantile(0.999))
max_rate  = int(x.max())

baseline = {
    "window_days": DAYS_WINDOW,
    "rate_p95": rate_p95,
    "rate_p99": rate_p99,
    "rate_p999": rate_p999,
    "max_per_minute": max_rate,
}
baseline

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_1772\3620823964.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_min = pd.read_sql(q, conn)


{'window_days': 30,
 'rate_p95': 77.25,
 'rate_p99': 144.0,
 'rate_p999': 411.22499999999945,
 'max_per_minute': 540}

In [3]:
df = pd.read_csv("artifacts/blocks_features.csv", parse_dates=["start","end"])


# Si tu df ya tiene prio1_share, perfecto. Si no, lo calculamos luego.
df.columns

Index(['start', 'end', 'n', 'duration_min', 'mean_rate', 'max_rate',
       'unique_tags', 'dominant_tag_share', 'unique_msgs',
       'dominant_msg_share', 'flood_candidate', 'flood_severity'],
      dtype='str')

In [4]:
def severity_from_rate(max_rate, base):
    if max_rate >= base["rate_p999"]:
        return "severe"
    if max_rate >= base["rate_p99"]:
        return "medium"
    return "none"

df["severity_v11"] = df["max_rate"].apply(lambda r: severity_from_rate(r, baseline))
df["severity_v11"].value_counts()

severity_v11
none      5342
medium     205
severe      19
Name: count, dtype: int64

In [5]:
def flood_candidate_v11(row, base):
    return (
        (row["n"] >= 50) and
        (row["max_rate"] >= base["rate_p99"]) and
        (row.get("prio1_share", 0.0) >= 0.80) and
        ((row["dominant_msg_share"] >= 0.70) or (row["dominant_tag_share"] >= 0.85))
    )

df["flood_candidate_v11"] = df.apply(lambda r: flood_candidate_v11(r, baseline), axis=1)
df["flood_candidate_v11"].value_counts()

flood_candidate_v11
False    5566
Name: count, dtype: int64

In [6]:
T_INFRA_TAGS = 200   # transversalidad
T_SUBSYS_TAGS = 50   # subsistema

INFRA_TOKENS = [
    "ENABLED", "BAD I/O", "I/O", "COMM", "NETWORK", "SERVER", "RESET", "LINK", "COMMUNICATION"
]

def is_infrastructure_message(msg):
    if msg is None or (isinstance(msg, float) and pd.isna(msg)):
        return False
    s = str(msg).upper()
    return any(tok in s for tok in INFRA_TOKENS)

def classify_type_v11(row, base):
    # Si no es candidato, no tipificamos como flood
    if not row["flood_candidate_v11"]:
        return "OTHER_OR_NO_FLOOD"

    ut = row["unique_tags"]
    dts = row["dominant_tag_share"]
    dms = row["dominant_msg_share"]
    sev = row["severity_v11"]
    pr1 = row.get("prio1_share", 0.0)

    # 1) INFRASTRUCTURE_EVENT
    # Nota: idealmente necesitamos el mensaje dominante real; por ahora usamos heurística adicional luego.
    # Como no tenemos "dominant_msg_text" en blocks_features, aplicamos solo estructura:
    if (dms >= 0.70) and (ut >= T_INFRA_TAGS) and (sev in ["medium","severe"]):
        return "INFRASTRUCTURE_EVENT"

    # 2) CHATTERING_POINT
    if (dts >= 0.95) and (sev in ["medium","severe"]):
        return "CHATTERING_POINT"

    # 3) LOCAL_PROCESS_INSTABILITY
    if (ut <= 15) and (dms >= 0.70) and (dts < 0.85) and (sev in ["medium","severe"]):
        return "LOCAL_PROCESS_INSTABILITY"

    # 4) SUBSYSTEM_TRIP_EVENT
    if (ut >= T_SUBSYS_TAGS) and (dms >= 0.50) and (pr1 >= 0.80) and (sev in ["medium","severe"]):
        return "SUBSYSTEM_TRIP_EVENT"

    return "OTHER_OR_NO_FLOOD"

df["flood_type_v11"] = df.apply(lambda r: classify_type_v11(r, baseline), axis=1)
df["flood_type_v11"].value_counts()

flood_type_v11
OTHER_OR_NO_FLOOD    5566
Name: count, dtype: int64

In [7]:
# Baseline sobre TODO el dataset (solo para validación histórica)

q = f"""
WITH minute_counts AS (
  SELECT
    DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0) AS [minute],
    COUNT(*) AS alarms
  FROM [{DB}].[{SCHEMA}].[{TABLE}]
  GROUP BY DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0)
)
SELECT alarms
FROM minute_counts;
"""

In [8]:
import pandas as pd
import numpy as np

q = f"""
WITH minute_counts AS (
  SELECT
    DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0) AS [minute],
    COUNT(*) AS alarms
  FROM [{DB}].[{SCHEMA}].[{TABLE}]
  GROUP BY DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0)
)
SELECT alarms
FROM minute_counts;
"""

df_min_all = pd.read_sql(q, conn)
x = df_min_all["alarms"].to_numpy()

baseline_all = {
    "scope": "all_history",
    "rate_p95": float(np.quantile(x, 0.95)),
    "rate_p99": float(np.quantile(x, 0.99)),
    "rate_p999": float(np.quantile(x, 0.999)),
    "max_per_minute": int(x.max()),
    "n_minutes": int(len(x)),
}
baseline_all

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_1772\3574549218.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_min_all = pd.read_sql(q, conn)


{'scope': 'all_history',
 'rate_p95': 102.0,
 'rate_p99': 180.0,
 'rate_p999': 345.0,
 'max_per_minute': 7734,
 'n_minutes': 138469}

In [9]:
def severity_from_rate(max_rate, base):
    if max_rate >= base["rate_p999"]:
        return "severe"
    if max_rate >= base["rate_p99"]:
        return "medium"
    return "none"

def flood_candidate_v11(row, base):
    return (
        (row["n"] >= 50) and
        (row["max_rate"] >= base["rate_p99"]) and
        (row.get("prio1_share", 0.0) >= 0.80) and
        ((row["dominant_msg_share"] >= 0.70) or (row["dominant_tag_share"] >= 0.85))
    )

T_INFRA_TAGS = 200
T_SUBSYS_TAGS = 50

def classify_type_v11(row, base):
    if not row["flood_candidate_v11"]:
        return "OTHER_OR_NO_FLOOD"

    ut = row["unique_tags"]
    dts = row["dominant_tag_share"]
    dms = row["dominant_msg_share"]
    sev = row["severity_v11"]
    pr1 = row.get("prio1_share", 0.0)

    if (dms >= 0.70) and (ut >= T_INFRA_TAGS) and (sev in ["medium","severe"]):
        return "INFRASTRUCTURE_EVENT"

    if (dts >= 0.95) and (sev in ["medium","severe"]):
        return "CHATTERING_POINT"

    if (ut <= 15) and (dms >= 0.70) and (dts < 0.85) and (sev in ["medium","severe"]):
        return "LOCAL_PROCESS_INSTABILITY"

    if (ut >= T_SUBSYS_TAGS) and (dms >= 0.50) and (pr1 >= 0.80) and (sev in ["medium","severe"]):
        return "SUBSYSTEM_TRIP_EVENT"

    return "OTHER_OR_NO_FLOOD"

# Aplica
df["severity_v11"] = df["max_rate"].apply(lambda r: severity_from_rate(r, baseline_all))
df["flood_candidate_v11"] = df.apply(lambda r: flood_candidate_v11(r, baseline_all), axis=1)
df["flood_type_v11"] = df.apply(lambda r: classify_type_v11(r, baseline_all), axis=1)

print("Baseline:", baseline_all)
print("\nFlood candidates:", df["flood_candidate_v11"].value_counts())
print("\nTypes:", df["flood_type_v11"].value_counts())

Baseline: {'scope': 'all_history', 'rate_p95': 102.0, 'rate_p99': 180.0, 'rate_p999': 345.0, 'max_per_minute': 7734, 'n_minutes': 138469}

Flood candidates: flood_candidate_v11
False    5566
Name: count, dtype: int64

Types: flood_type_v11
OTHER_OR_NO_FLOOD    5566
Name: count, dtype: int64


In [10]:
DAYS_WINDOW = 30

mx = pd.read_sql(
    f"SELECT MAX(ALARMDATETIME) AS mx_t FROM [{DB}].[{SCHEMA}].[{TABLE}];",
    conn
)
mx_t = pd.to_datetime(mx.loc[0, "mx_t"])
start_window = mx_t - pd.Timedelta(days=DAYS_WINDOW)

print("mx_t:", mx_t)
print("window_start:", start_window)

df_recent = df[(df["start"] >= start_window) & (df["start"] <= mx_t)].copy()
df_recent.shape

mx_t: 2025-12-10 23:57:09
window_start: 2025-11-10 23:57:09


C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_1772\317081799.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mx = pd.read_sql(


(420, 15)

In [11]:
q = f"""
WITH mx AS (
  SELECT MAX(ALARMDATETIME) AS mx_t
  FROM [{DB}].[{SCHEMA}].[{TABLE}]
),
minute_counts AS (
  SELECT
    DATEADD(minute, DATEDIFF(minute, 0, a.ALARMDATETIME), 0) AS [minute],
    COUNT(*) AS alarms
  FROM [{DB}].[{SCHEMA}].[{TABLE}] a
  CROSS JOIN mx
  WHERE a.ALARMDATETIME >= DATEADD(day, -{DAYS_WINDOW}, mx.mx_t)
  GROUP BY DATEADD(minute, DATEDIFF(minute, 0, a.ALARMDATETIME), 0)
)
SELECT alarms
FROM minute_counts;
"""
df_min_30 = pd.read_sql(q, conn)
x = df_min_30["alarms"].to_numpy()

baseline_30 = {
    "scope": f"last_{DAYS_WINDOW}_days",
    "rate_p95": float(np.quantile(x, 0.95)),
    "rate_p99": float(np.quantile(x, 0.99)),
    "rate_p999": float(np.quantile(x, 0.999)),
    "max_per_minute": int(x.max()),
    "window_days": DAYS_WINDOW,
}
baseline_30

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_1772\2576286431.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_min_30 = pd.read_sql(q, conn)


{'scope': 'last_30_days',
 'rate_p95': 77.25,
 'rate_p99': 144.0,
 'rate_p999': 411.22499999999945,
 'max_per_minute': 540,
 'window_days': 30}

In [12]:
df_recent["severity_v11"] = df_recent["max_rate"].apply(lambda r: severity_from_rate(r, baseline_30))
df_recent["flood_candidate_v11"] = df_recent.apply(lambda r: flood_candidate_v11(r, baseline_30), axis=1)
df_recent["flood_type_v11"] = df_recent.apply(lambda r: classify_type_v11(r, baseline_30), axis=1)

print("Baseline:", baseline_30)
print("\nFlood candidates:", df_recent["flood_candidate_v11"].value_counts())
print("\nTypes:", df_recent["flood_type_v11"].value_counts())

Baseline: {'scope': 'last_30_days', 'rate_p95': 77.25, 'rate_p99': 144.0, 'rate_p999': 411.22499999999945, 'max_per_minute': 540, 'window_days': 30}

Flood candidates: flood_candidate_v11
False    420
Name: count, dtype: int64

Types: flood_type_v11
OTHER_OR_NO_FLOOD    420
Name: count, dtype: int64


In [13]:
# === Flood D v1.1 (híbrido) - análisis completo en UNA celda ===
# Requiere que ya tengas: conn (pyodbc), y variables DB, SCHEMA, TABLE definidas.
# Requiere archivo: artifacts/blocks_features.csv

import pandas as pd
import numpy as np

# -----------------------------
# Config
# -----------------------------
CSV_PATH = "artifacts/blocks_features.csv"

# Baseline: usa all-history para validar reglas contra todo el dataset
BASELINE_SCOPE = "all_history"   # "all_history" (validación) | "last_N_days" (producción-like)
DAYS_WINDOW = 30                 # si usas last_N_days

# Flood candidate params (v1.1)
N_MIN = 50
DOM_MSG_TH = 0.70
DOM_TAG_TH = 0.85
PRIO1_TH = 0.80

# Typology thresholds
T_INFRA_TAGS = 200
T_SUBSYS_TAGS = 50

# -----------------------------
# Helpers
# -----------------------------
def get_baseline_all_history(conn, DB, SCHEMA, TABLE):
    q = f"""
    WITH minute_counts AS (
      SELECT
        DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0) AS [minute],
        COUNT(*) AS alarms
      FROM [{DB}].[{SCHEMA}].[{TABLE}]
      GROUP BY DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0)
    )
    SELECT alarms
    FROM minute_counts;
    """
    df_min = pd.read_sql(q, conn)
    x = df_min["alarms"].to_numpy()
    return {
        "scope": "all_history",
        "rate_p95": float(np.quantile(x, 0.95)),
        "rate_p99": float(np.quantile(x, 0.99)),
        "rate_p999": float(np.quantile(x, 0.999)),
        "max_per_minute": int(x.max()),
        "n_minutes": int(len(x)),
    }

def get_baseline_last_days(conn, DB, SCHEMA, TABLE, days_window):
    q = f"""
    WITH mx AS (
      SELECT MAX(ALARMDATETIME) AS mx_t
      FROM [{DB}].[{SCHEMA}].[{TABLE}]
    ),
    minute_counts AS (
      SELECT
        DATEADD(minute, DATEDIFF(minute, 0, a.ALARMDATETIME), 0) AS [minute],
        COUNT(*) AS alarms
      FROM [{DB}].[{SCHEMA}].[{TABLE}] a
      CROSS JOIN mx
      WHERE a.ALARMDATETIME >= DATEADD(day, -{days_window}, mx.mx_t)
      GROUP BY DATEADD(minute, DATEDIFF(minute, 0, a.ALARMDATETIME), 0)
    )
    SELECT alarms
    FROM minute_counts;
    """
    df_min = pd.read_sql(q, conn)
    x = df_min["alarms"].to_numpy()
    return {
        "scope": f"last_{days_window}_days",
        "window_days": days_window,
        "rate_p95": float(np.quantile(x, 0.95)),
        "rate_p99": float(np.quantile(x, 0.99)),
        "rate_p999": float(np.quantile(x, 0.999)),
        "max_per_minute": int(x.max()),
        "n_minutes": int(len(x)),
    }

def severity_from_rate(max_rate, base):
    if max_rate >= base["rate_p999"]:
        return "severe"
    if max_rate >= base["rate_p99"]:
        return "medium"
    return "none"

def fetch_prio1_share(conn, DB, SCHEMA, TABLE, start, end):
    q = f"""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN PRIORITY = 1 THEN 1 ELSE 0 END) AS prio1
    FROM [{DB}].[{SCHEMA}].[{TABLE}]
    WHERE ALARMDATETIME BETWEEN ? AND ?;
    """
    r = pd.read_sql(q, conn, params=[start, end]).iloc[0]
    total = int(r["total"])
    prio1 = int(r["prio1"]) if pd.notna(r["prio1"]) else 0
    return (prio1 / total) if total > 0 else 0.0

def flood_candidate_v11(row, base):
    return (
        (row["n"] >= N_MIN) and
        (row["max_rate"] >= base["rate_p99"]) and
        (row["prio1_share"] >= PRIO1_TH) and
        ((row["dominant_msg_share"] >= DOM_MSG_TH) or (row["dominant_tag_share"] >= DOM_TAG_TH))
    )

def classify_type_v11(row, base):
    if not row["flood_candidate_v11"]:
        return "OTHER_OR_NO_FLOOD"

    ut = row["unique_tags"]
    dts = row["dominant_tag_share"]
    dms = row["dominant_msg_share"]
    sev = row["severity_v11"]
    pr1 = row["prio1_share"]

    # Precedencia:
    if (dms >= DOM_MSG_TH) and (ut >= T_INFRA_TAGS) and (sev in ["medium", "severe"]):
        return "INFRASTRUCTURE_EVENT"

    if (dts >= 0.95) and (sev in ["medium", "severe"]):
        return "CHATTERING_POINT"

    if (ut <= 15) and (dms >= DOM_MSG_TH) and (dts < 0.85) and (sev in ["medium", "severe"]):
        return "LOCAL_PROCESS_INSTABILITY"

    if (ut >= T_SUBSYS_TAGS) and (dms >= 0.50) and (pr1 >= PRIO1_TH) and (sev in ["medium", "severe"]):
        return "SUBSYSTEM_TRIP_EVENT"

    return "OTHER_OR_NO_FLOOD"


# -----------------------------
# 1) Load blocks features
# -----------------------------
df = pd.read_csv(CSV_PATH, parse_dates=["start", "end"])

# -----------------------------
# 2) Baseline
# -----------------------------
if BASELINE_SCOPE == "all_history":
    baseline = get_baseline_all_history(conn, DB, SCHEMA, TABLE)
else:
    baseline = get_baseline_last_days(conn, DB, SCHEMA, TABLE, DAYS_WINDOW)

print("=== Baseline ===")
print(baseline)

# -----------------------------
# 3) Pre-candidates (sin prio1) para minimizar queries SQL
# -----------------------------
pre = df[
    (df["n"] >= N_MIN) &
    (df["max_rate"] >= baseline["rate_p99"]) &
    (
        (df["dominant_msg_share"] >= DOM_MSG_TH) |
        (df["dominant_tag_share"] >= DOM_TAG_TH)
    )
].copy()

print("\n=== Pre-candidates (sin prio1) ===")
print("count:", len(pre))

# -----------------------------
# 4) Compute prio1_share SOLO para pre-candidates
# -----------------------------
pre["prio1_share"] = [
    fetch_prio1_share(conn, DB, SCHEMA, TABLE, s, e)
    for s, e in zip(pre["start"], pre["end"])
]

# Merge back
df = df.merge(pre[["start", "end", "prio1_share"]], on=["start", "end"], how="left")
df["prio1_share"] = df["prio1_share"].fillna(0.0)

print("\n=== prio1_share stats (all blocks) ===")
print(df["prio1_share"].describe())

# -----------------------------
# 5) Apply v1.1 rules
# -----------------------------
df["severity_v11"] = df["max_rate"].apply(lambda r: severity_from_rate(r, baseline))
df["flood_candidate_v11"] = df.apply(lambda r: flood_candidate_v11(r, baseline), axis=1)
df["flood_type_v11"] = df.apply(lambda r: classify_type_v11(r, baseline), axis=1)

print("\n=== Severity (v1.1) ===")
print(df["severity_v11"].value_counts())

print("\n=== Flood candidates (v1.1) ===")
print(df["flood_candidate_v11"].value_counts())

print("\n=== Flood types (v1.1) ===")
print(df["flood_type_v11"].value_counts())

# -----------------------------
# 6) Show top examples per type for quick validation
# -----------------------------
cols_show = ["start","end","n","duration_min","max_rate","unique_tags","dominant_msg_share","dominant_tag_share","prio1_share","severity_v11","flood_candidate_v11","flood_type_v11"]
print("\n=== Top 5 per flood_type_v11 by max_rate ===")
for t in ["INFRASTRUCTURE_EVENT","CHATTERING_POINT","LOCAL_PROCESS_INSTABILITY","SUBSYSTEM_TRIP_EVENT"]:
    sub = df[df["flood_type_v11"] == t].sort_values("max_rate", ascending=False).head(5)
    print(f"\n--- {t} (n={len(df[df['flood_type_v11']==t])}) ---")
    if len(sub) == 0:
        print("No rows.")
    else:
        print(sub[cols_show].to_string(index=False))

# -----------------------------
# 7) Save updated features
# -----------------------------
out_path = "artifacts/blocks_features_v11_with_prio1.csv"
df.to_csv(out_path, index=False)
print("\nGuardado:", out_path)

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_1772\4144645931.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_min = pd.read_sql(q, conn)


=== Baseline ===
{'scope': 'all_history', 'rate_p95': 102.0, 'rate_p99': 180.0, 'rate_p999': 345.0, 'max_per_minute': 7734, 'n_minutes': 138469}

=== Pre-candidates (sin prio1) ===
count: 70


C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_1772\4144645931.py:98: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  r = pd.read_sql(q, conn, params=[start, end]).iloc[0]
C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_1772\4144645931.py:98: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  r = pd.read_sql(q, conn, params=[start, end]).iloc[0]
C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_1772\4144645931.py:98: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  r = pd.read_sql(q, conn, params=[start, end]).iloc[0]
C:\Users\Andr


=== prio1_share stats (all blocks) ===
count    5566.000000
mean        0.011833
std         0.105721
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: prio1_share, dtype: float64

=== Severity (v1.1) ===
severity_v11
none      5403
medium     134
severe      29
Name: count, dtype: int64

=== Flood candidates (v1.1) ===
flood_candidate_v11
False    5498
True       68
Name: count, dtype: int64

=== Flood types (v1.1) ===
flood_type_v11
OTHER_OR_NO_FLOOD            5522
LOCAL_PROCESS_INSTABILITY      27
SUBSYSTEM_TRIP_EVENT           10
CHATTERING_POINT                6
INFRASTRUCTURE_EVENT            1
Name: count, dtype: int64

=== Top 5 per flood_type_v11 by max_rate ===

--- INFRASTRUCTURE_EVENT (n=1) ---
              start                 end     n  duration_min  max_rate  unique_tags  dominant_msg_share  dominant_tag_share  prio1_share severity_v11  flood_candidate_v11       flood_type_v11
2025-05-03 09:07:08 2025-05-03

In [14]:
print(df["flood_type_v11"].value_counts())
print("\nSeveridad dentro de candidatos:")
print(pd.crosstab(df["flood_type_v11"], df["severity_v11"]))

flood_type_v11
OTHER_OR_NO_FLOOD            5522
LOCAL_PROCESS_INSTABILITY      27
SUBSYSTEM_TRIP_EVENT           10
CHATTERING_POINT                6
INFRASTRUCTURE_EVENT            1
Name: count, dtype: int64

Severidad dentro de candidatos:
severity_v11               medium  none  severe
flood_type_v11                                 
CHATTERING_POINT                5     0       1
INFRASTRUCTURE_EVENT            0     0       1
LOCAL_PROCESS_INSTABILITY      21     0       6
OTHER_OR_NO_FLOOD              98  5403      21
SUBSYSTEM_TRIP_EVENT           10     0       0
